In [1]:
!pip install elasticsearch==8.19.1 kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.5/940.5 kB 1.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [elasticsearch]0m [elasticsearch]ort]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [13]:

from elasticsearch import Elasticsearch
import os
import pandas as pd



In [8]:
!docker cp search-system-es01-1:/usr/share/elasticsearch/config/certs/ca/ca.crt ./ca.crt

In [8]:
!docker info | grep -i memory
!docker info | grep -i cpu

 Total Memory: 7.758GiB
 CPUs: 2


In [9]:

!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       5.9Gi       155Mi        63Mi       2.3Gi       1.9Gi
Swap:             0B          0B          0B


In [14]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay          32G   21G  9.0G  70% /
tmpfs            64M     0   64M   0% /dev
shm              64M  4.0K   64M   1% /dev/shm
/dev/root        29G   22G  7.7G  74% /vscode
/dev/loop4       32G   21G  9.0G  70% /workspaces
/dev/sdb1        44G  4.0G   38G  10% /tmp


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("phamtheds/news-dataset-vietnameses")
print("Path to dataset files:", path)

D:\program files\miniconda\envs\search-db\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
D:\program files\miniconda\envs\search-db\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 32%|███▏      | 100M/308M [00:19<00:41, 5.28MB/s] 


KeyboardInterrupt: 

In [22]:
client = Elasticsearch(
    hosts=["https://localhost:9200"],  # Địa chỉ Elasticsearch
    basic_auth=("elastic", "elastic"),
    request_timeout=60,
    ca_certs="./ca.crt"
)
client.info()

ObjectApiResponse({'name': 'es01', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Ciqvk2H9SR-35cdh13Nbjg', 'version': {'number': '8.19.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'aa0a7826e719b392e7782716b323c4fb8fa3b392', 'build_date': '2025-09-16T22:06:03.940754111Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [3]:

def get_all_file_names(folder_path):
    try:
        # List all files in the folder and remove ".json" extension
        file_names = [
            os.path.splitext(file)[0] for file in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, file)) and (file.endswith(".ndjson") or file.endswith(".csv"))
        ]
        return file_names
    except FileNotFoundError:
        print(f"The folder '{folder_path}' does not exist.")
        return []

folder_path = path
# folder_path = "data"
file_names = get_all_file_names(folder_path)
print("Files in folder:", file_names)

NameError: name 'path' is not defined

In [32]:
template={
    "index_patterns": [
        "articles*"
    ],
    "template": {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "60s",
            "translog.durability": "async",
            "translog.sync_interval": "30s",
            "merge.scheduler.max_thread_count": 1,
            "indexing.slowlog.threshold.index.warn": "10s",
            "indexing.slowlog.threshold.index.info": "5s",
            "analysis": {
                "analyzer": {

                    "nfd_normalized": {
                        "tokenizer": "icu_tokenizer",
                        "char_filter": [
                            "nfd_normalizer"
                        ]
                    }
                },
                "char_filter": {
                    "nfd_normalizer": {
                        "type": "icu_normalizer",
                        "name": "nfc",
                        "mode": "decompose"
                    }
                }

            }
        },
        "mappings": {
            "properties": {
                "url": {
                    "type": "keyword"
                },
                "title": {
                    "type": "text",
                },
                "summary": {
                    "type": "text"
                },
                "contents": {
                    "type": "text",
                    "analyzer": "whitespace"
                },
                "date": {
                    "type": "date"
                },
                "authors": {
                    "type": "text",
                    "fields": {
                        "keyword": {
                            "type": "keyword"
                        }
                    }
                },
                "category": {
                    "type": "keyword"
                },
                "tags": {
                    "type": "text",
                    "fields": {
                        "keyword": {
                            "type": "keyword"
                        }
                    }
                }
            }
        }
    }
}

In [33]:
# Tạo index template
import json
with open('index_template.json', 'r') as f:
    # template = json.load(f)
    template = template
if client.indices.exists_index_template(name="baolaodong_template"):
    client.indices.delete_index_template(name="baolaodong_template")
client.indices.put_index_template(
    name="baolaodong_template",
    index_patterns=template['index_patterns'],
    template=template['template']
)

ObjectApiResponse({'acknowledged': True})

In [9]:
import ast
import re
def safe_literal_eval(val):
    """Chuyển đổi string list thành Python list an toàn"""
    if pd.isna(val) or val == '':
        return []
    try:
        # Xử lý trường hợp có dấu ngoặc vuông
        if val.startswith('[') and val.endswith(']'):
            return ast.literal_eval(val)
        else:
            # Nếu không phải list, coi như single value
            return [val.strip()]
    except (ValueError, SyntaxError):
        # Nếu có lỗi, split bằng dấu phẩy
        return [item.strip().strip("'\"") for item in val.split(',')]

In [10]:
data = pd.read_csv("data/Dataset_articles_NoID.csv")

In [11]:
data.head()

,URL,Title,Summary,Contents,Date,Author(s),Category,Tags
0,https://laodong.vn/bat-dong-san/thong-tin-ngoc...,"Thông tin “Ngọc Trinh mua đất ở Bảo Lộc"" chỉ l...","Lâm Đồng - Lãnh đạo thành phố Bảo Lộc, Lâm Đồn...","Những ngày vừa qua, trên trang Facebook chính ...","Thứ sáu, 20/05/2022 08:56 (GMT+7)",Phương Nhiên,Bất động sản,"['Lâm Đồng', 'Ngọc Trinh', 'Chiêu trò', 'Giá đ..."
1,https://laodong.vn/bat-dong-san/lo-hong-trong-...,Lỗ hổng trong việc thẩm tra năng lực tài chính...,TPHCM - Việc không thể cưỡng chế thuế của hai ...,"Theo thông tin từ Cục Thuế TP.HCM, hiện cơ qua...","Thứ sáu, 20/05/2022 08:10 (GMT+7)",Gia Miêu,Bất động sản,"['Thủ Thiêm', 'Đấu giá đất']"
2,https://laodong.vn/bat-dong-san/som-hoan-thien...,Sớm hoàn thiện các dự án nhà ở xã hội để CNLĐ ...,"Hiện trên địa bàn tỉnh Ninh Bình có 32 khu, cụ...",CNLĐ mong muốn sớm được tiếp cận với nhà ở xã ...,"Thứ sáu, 20/05/2022 07:47 (GMT+7)",NGUYỄN TRƯỜNG,Bất động sản,"['Dự án', 'Nhà ở xã hội', 'Dự án nhà ở xã hội'..."
3,https://laodong.vn/bat-dong-san/chi-tiet-ho-so...,Chi tiết hồ sơ hoàn công nhà ở năm 2022,Hoàn công nhà ở với ý nghĩa là điều kiện để đư...,Hoàn công nhà ở là một thủ tục hành chính tron...,"Thứ sáu, 20/05/2022 06:44 (GMT+7)",Kim Nhung (T/H),Bất động sản,"['Giấy phép xây dựng', 'Hồ sơ hoàn công', 'nhà..."
4,https://laodong.vn/bat-dong-san/khoi-tao-khong...,"Khởi tạo không gian sống đẳng cấp, đón sóng đầ...",Có rất nhiều lý do khiến những dự án thấp nội ...,Đi dọc đường Lê Văn Lương kéo dài xuống khu Dư...,"Thứ năm, 19/05/2022 15:30 (GMT+7)",Huyền Nguyễn,Bất động sản,['An Quý Villa']


In [29]:
from elasticsearch import helpers
import traceback
# Process NDJSON files

# Create index (matches wikipedia-people* pattern)

INDEX_NAME = "articles-csv"
if not client.indices.exists(index=INDEX_NAME):
    client.indices.create(index=INDEX_NAME)
    print(f"Index '{INDEX_NAME}' created successfully")

batch_size = 5
actions = []


count =0
for index, row in data.iterrows():

    # Xử lý tags từ string list thành Python list
    tags = safe_literal_eval(row['Tags'])

    # Xử lý authors tương tự
    authors = safe_literal_eval(row['Author(s)'])


    # Sử dụng giá trị mặc định nếu thiếu
    # if not full_text or not full_text.strip():
    #     full_text = "No content available"

    if count >10:
        break

    count +=1

    # Prepare ES document
    es_doc = {
        "_index": INDEX_NAME,
        "_id": row['URL'],
        "_source": {
            "url": row['URL'],
            "title": row['Title'],
            "summary": row['Summary'],
            "contents": row['Contents'],
            "date": row['Date'],
            "authors": authors,
            "category": row['Category'],
            "tags": tags
        }
    }
    actions.append(es_doc)

    # Bulk index when batch is full
    if len(actions) >= batch_size:
        # try:
            helpers.bulk(client, actions)
            # print(f"Indexed {len(actions)} documents from {file_names[5]}")
            actions = []


        # except Exception as e:
        #     print(f"Error indexing batch: {e}")
        #     print(actions)

# Index any remaining documents
# if actions:
#     try:
#         helpers.bulk(client, actions)
#         print(f"Indexed final {len(actions)} documents")
#     except Exception as e:
#         print(f"Error indexing final batch: {e}")
#         print(actions)

print("Ingestion complete!")

Ingestion complete!
